# 06 — Per-fold dataset config overrides

`FoldConfigFactory.fold_dataset_config` takes the shared training dataset configuration and overrides only its `split_regions` with the regions of the requested fold. This notebook confirms that each fold receives a distinct `split_regions` while all other dataset fields stay identical across folds.

Because the real factory reads the dataset layout from `/ste/rnd`, which is not mounted, we reproduce the exact override logic on a synthetic `DatasetConfiguration` whose `split_regions` we replace with each fold's plan. The fold regions themselves come from the real `FoldPlanner`.

Modules exercised: `pipelines.cross_validation_pipeline.folds.FoldPlanner`, `configuration.dataset_config.DatasetConfiguration`, `tools.regions.SplitRegions`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path("../..").resolve()))

import numpy             as np
import matplotlib.pyplot as plt

try:
    import torch
    torch.manual_seed(0)
except Exception:
    torch = None

SEED = 0
RNG  = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.dpi"     : 110,
    "font.size"      : 10,
    "axes.titlesize" : 11,
    "axes.labelsize" : 10,
    "image.cmap"     : "viridis",
})

print("Bootstrap complete. Repository root on sys.path:", Path("../..").resolve())


## Build a synthetic base dataset config and the fold planner

The base config mirrors the fields the real `training_dataset_config` sets; only `split_regions` will differ per fold.

In [ ]:
from dataclasses import replace

from configuration.cross_validation_config import CrossValidationConfig
from configuration.dataset_config         import DatasetConfiguration, PatchConfiguration, InputConfig, Representation
from pipelines.cross_validation_pipeline.folds import FoldPlanner
from tools.regions import CropRegion, SplitRegions

AZ_START, AZ_END = 0, 500
RG_START, RG_END = 0, 100

config                     = CrossValidationConfig()
config.folds.n_folds       = 5
config.folds.azimuth_start = AZ_START
config.folds.azimuth_end   = AZ_END

planner = FoldPlanner(config, range_start=RG_START, range_end=RG_END)

placeholder = SplitRegions(
    train = CropRegion(AZ_START, AZ_END, RG_START, RG_END),
    val   = CropRegion(AZ_START, AZ_END, RG_START, RG_END),
    test  = CropRegion(AZ_START, AZ_END, RG_START, RG_END),
)

base_config = DatasetConfiguration(
    preprocessing_run_directory = Path("/synthetic/dataset"),
    parameters_path             = Path("/synthetic/params.npy"),
    split_regions               = placeholder,
    patch                       = PatchConfiguration(size=(64, 64), stride=32),
    input_config                = InputConfig(),
    batch_size                  = 256,
)

print("base batch_size  :", base_config.batch_size)
print("base patch size  :", base_config.patch.size)
print("base patch stride:", base_config.patch.stride)

## Apply the per-fold override

This mirrors `FoldConfigFactory.fold_dataset_config`: copy the base config, then set `split_regions` to the fold's regions. We keep the per-fold configs side by side for inspection.

In [ ]:
fold_configs = []
for fold_index in range(config.folds.n_folds):
    fold_config               = replace(base_config)
    fold_config.split_regions = planner.plan(fold_index).split_regions
    fold_configs.append(fold_config)

for fold_index, fold_config in enumerate(fold_configs):
    test_region = fold_config.split_regions.regions("test")[0]
    val_region  = fold_config.split_regions.regions("val")[0]
    print(f"fold {fold_index}: test=[{test_region.azimuth_start}, {test_region.azimuth_end})  val=[{val_region.azimuth_start}, {val_region.azimuth_end})")

## Confirm only split_regions varies across folds

Every non-region field must be identical to the base config; the `split_regions` field must differ for every fold.

In [ ]:
shared_fields = ["preprocessing_run_directory", "parameters_path", "batch_size"]

for name in shared_fields:
    values = {getattr(cfg, name) for cfg in fold_configs}
    print(f"{name:30s} constant across folds: {len(values) == 1}")

region_signatures = [cfg.split_regions.regions("test")[0].as_tuple() for cfg in fold_configs]
print("distinct test regions per fold:", len(set(region_signatures)) == len(fold_configs))

## Visualise the per-fold test region overrides

A bar per fold marks where the test region sits in azimuth, confirming each fold config carries a different test window while sharing the same surrounding configuration.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 0.6 * len(fold_configs) + 1))

for fold_index, fold_config in enumerate(fold_configs):
    region = fold_config.split_regions.regions("test")[0]
    ax.barh(fold_index, region.azimuth_end - region.azimuth_start, left=region.azimuth_start, color="#d96c6c", edgecolor="white")

ax.set_xlim(AZ_START, AZ_END)
ax.set_yticks(range(len(fold_configs)))
ax.set_yticklabels([f"fold {i}" for i in range(len(fold_configs))])
ax.invert_yaxis()
ax.set_xlabel("azimuth (samples)")
ax.set_title("Per-fold test region override")
plt.show()

## Expected visual outcome

The shared fields report `constant across folds = True`, the test regions are all distinct, and the bar chart shows the red test window stepping across the azimuth axis from fold to fold. This is the override behaviour of `fold_dataset_config` reproduced exactly.